# Install & Import

In [ ]:
# GPU Checker only
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
!pip install transformers scikit-learn pandas torch -q
import pandas as pd
from sklearn.model_selection import train_test_split
print("Libraries ready")

# Load & Map Labels

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
CSV_PATH = "full.csv"
df = pd.read_csv(CSV_PATH)

# Standardize column name for downstream compatibility
df = df.rename(columns={"article": "text"})

# Labels are already 0 and 1, cast to integer to prevent type warnings
df["label"] = df["label"].astype(int)

df = df.dropna(subset=["text", "label"])
print(f"Rows: {len(df)} | Columns: {df.columns.tolist()}")
print(df["label"].value_counts())

# Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print("Train distribution:", train_df["label"].value_counts().to_dict())
print("Test distribution:", test_df["label"].value_counts().to_dict())

# Tokenization & Dataset Conversion

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512 # if the device can do higher than this, use 256 or 512
    )

# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Apply tokenization in batches for speed
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Remove raw text column to free memory
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

print("Tokenization complete.")
print(f"Train samples: {len(tokenized_train)} | Test samples: {len(tokenized_test)}")

# Training Setup & Config A (Full Fine-tuning)

In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load model with classification head
MODEL_NAME = "FacebookAI/xlm-roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Training arguments for Config A (Full fine-tuning)
training_args = TrainingArguments(
    output_dir="./results_configA",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    learning_rate=2e-5,
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

# Start training
print("Starting Config A training (Full fine-tuning)...")
trainer.train()

# Evaluate on test set
print("\nEvaluating on test set...")
results = trainer.evaluate()
print(f"Config A Results: {results}")

# Config B (Freeze Base + Train Classifier Head)

In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load model with classification head
MODEL_NAME = "FacebookAI/xlm-roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# FREEZE the base model - only train the classifier head
for param in model.base_model.parameters():
    param.requires_grad = False

print("Frozen parameters:", sum(p.numel() for p in model.base_model.parameters() if not p.requires_grad))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Training arguments for Config B
training_args = TrainingArguments(
    output_dir="./results_configB",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_configB",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    learning_rate=2e-5,
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

# Start training
print("Starting Config B training (Freeze base + train head only)...")
trainer.train()

# Evaluate on test set
print("\nEvaluating on test set...")
results = trainer.evaluate()
print(f"Config B Results: {results}")

# Config C (LoRA)

In [ ]:
# Install LoRA library if not present
!pip install peft -q

import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

# 1. Load Model
MODEL_NAME = "FacebookAI/xlm-roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 2. Setup LoRA Config
# This freezes the main model and adds small adapter layers
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,            # Rank of LoRA update matrices
    lora_alpha=32,   # Scaling factor
    target_modules=["key", "query", "value"], # Modules to apply LoRA
    lora_dropout=0.05,
    bias="none"
)

# 3. Apply LoRA to Model
model = get_peft_model(model, peft_config)

# Check trainable parameters (should be very low, ~1-2%)
model.print_trainable_parameters()

# 4. Compute Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./results_configC",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_configC",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    learning_rate=2e-5,
    report_to="none"
)

# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

# 7. Train
print("Starting Config C training (LoRA)...")
trainer.train()

print("\nEvaluating on test set...")
results = trainer.evaluate()
print(f"Config C Results: {results}")

# Summary

Experiment 1 evaluated three fine-tuning configurations on `xlm-roberta-base` using the 3,206-entry Fake News Filipino dataset. `Config A (full fine-tuning)` achieved 96.99% accuracy and 96.98% F1, successfully learning the stylistic differences between real and fake articles. `Config B (frozen base)` dropped to 59.25% accuracy and 70.96% F1, confirming that preventing stylistic adaptation severely limits performance. `Config C (LoRA)` reached 81.81% accuracy and 83.96% F1, offering faster training at the cost of reduced accuracy. Full fine-tuning remains optimal for this task, while LoRA provides a viable lightweight fallback for deployment-constrained environments.